# FYP — Audio Preprocessing & Clip Curation (v1)

This notebook builds **training clips** from your downloaded Xeno-Canto recordings, using a simple pipeline that mimics deployment:

1. Decode → mono → resample (model sample rate)
2. Cap long recordings by selecting a 30s region (coarse scan)
3. Split into consecutive 3s windows
4. **RMS gate** only to remove silence / extremely faint windows
5. Run **BirdNET-Lite** as a *teacher* to label windows as:
   - target **species** (keep)
   - **non_bird** (keep)
   - **wrong_bird** (drop)
6. Per recording: select (seeded) up to **3** species clips + **1–2** non-bird clips (lowest bird confidence)
7. Save only selected clips + write manifests.

**Folder layout used (recommended):**
- Source audio: `src/dataset/bird_data/raw/<species_label>/XC<id>.mp3`
- Input manifests: `src/dataset/bird_data/manifests/<species_label>_downloaded.csv`
- Output clips: `src/dataset/bird_data/clips_v1/species/<species_label>/...` and `.../non_bird/<species_label>/...`
- Output manifests: `src/dataset/bird_data/manifests/clips_v1/<species_label>_clips.csv`


## 0) Install dependencies (once)

You'll need:
- `ffmpeg` available on PATH (MP3 decoding)
- `numpy`, `pandas`, `librosa`, `soundfile`
- `birdnetlib` (Python API for BirdNET-Lite / BirdNET-Analyzer) citeturn0search2turn0search9


In [1]:
# If you haven't installed these in your venv, uncomment:
%pip install -U numpy pandas librosa soundfile birdnetlib tqdm

import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())


Note: you may need to restart the kernel to use updated packages.
Python: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.26100-SP0


## 1) Config
Start with defaults; tune later if needed.


In [2]:
from pathlib import Path
import sys

# --- Find repo root by locating src/config.py (NOT just src/) ---
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "config.py").exists():
    repo_root = repo_root.parent
assert (repo_root / "src" / "config.py").exists(), f"Couldn't find src/config.py from {Path.cwd()}"

# Ensure repo_root is first on sys.path so the correct `src` is imported
sys.path.insert(0, str(repo_root))

from src.config import CONFIG

# --- Dataset root: folder that contains bird_data/ ---
# If you run notebook from .../src/dataset, this is CWD.
# If you run from repo root, fall back to repo_root/src/dataset.
cwd = Path.cwd().resolve()
if (cwd / CONFIG.paths.data_dir).exists():
    DATASET_ROOT = cwd
else:
    DATASET_ROOT = (repo_root / "src" / "dataset").resolve()
    assert (DATASET_ROOT / CONFIG.paths.data_dir).exists(), f"Can't find {CONFIG.paths.data_dir} under {DATASET_ROOT}"

BIRD_DATA_DIR = DATASET_ROOT / CONFIG.paths.data_dir
RAW_DIR = BIRD_DATA_DIR / CONFIG.paths.raw_dir
MANIFESTS_DIR = BIRD_DATA_DIR / CONFIG.paths.manifests_dir

# ---- AUDIO CONSTANTS (globals) ----
SR_MODEL = 16000

# BirdNET-Lite runs at 48k internally; we resample clips to it for teacher scoring
SR_TEACHER = 48000

CLIP_LEN_S = 3.0
STRIDE_S = 3.0
SKIP_FIRST_S = 1.0
INCLUDE_ZERO_WINDOW = True

CAP_RECORDING_S = 30.0
CAP_STEP_S = 5.0

RMS_KEEP_PERCENTILE = 30.0   # within-recording gating percentile
RMS_ABS_MIN_DB = -40.0       # absolute minimum RMS in dBFS
EPS = 1e-10

BIRD_CONF_THR = 0.15         # teacher detection threshold
SPECIES_CONF_THR = 0.25      # target species confidence threshold

MAX_SPECIES_CLIPS_PER_REC = 3
NONBIRD_CLIPS_PER_REC = 2
SEED = 12345

# ---- Versioned outputs ----
CLIPS_VERSION = "clips_v1"
OUT_CLIPS_DIR = BIRD_DATA_DIR / CLIPS_VERSION
OUT_SPECIES_DIR = OUT_CLIPS_DIR / "species"
OUT_NONBIRD_DIR = OUT_CLIPS_DIR / "non_bird"
OUT_CLIP_MANIFESTS_DIR = MANIFESTS_DIR / CLIPS_VERSION

OUT_SPECIES_DIR.mkdir(parents=True, exist_ok=True)
OUT_NONBIRD_DIR.mkdir(parents=True, exist_ok=True)
OUT_CLIP_MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

print("CWD:", Path.cwd())
print("REPO_ROOT:", repo_root)
print("DATASET_ROOT:", DATASET_ROOT)
print("BIRD_DATA_DIR:", BIRD_DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)
print("OUT_CLIPS_DIR:", OUT_CLIPS_DIR)
print("OUT_CLIP_MANIFESTS_DIR:", OUT_CLIP_MANIFESTS_DIR)

def resolve_source_path(local_path: str) -> Path:
    """
    CSV local_path like:
      bird_data\\raw\\accipiter_nisus\\XC1015215.mp3
    Resolve relative to DATASET_ROOT (src/dataset).
    """
    p = Path(str(local_path).replace("\\", "/"))
    return p if p.is_absolute() else (DATASET_ROOT / p).resolve()


CWD: c:\Users\shado\Year3Projects\FYP\src\dataset
REPO_ROOT: C:\Users\shado\Year3Projects\FYP
DATASET_ROOT: C:\Users\shado\Year3Projects\FYP\src\dataset
BIRD_DATA_DIR: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data
RAW_DIR: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\raw
MANIFESTS_DIR: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\manifests
OUT_CLIPS_DIR: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\clips_v1
OUT_CLIP_MANIFESTS_DIR: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\manifests\clips_v1


## 2) BirdNET-Lite teacher setup

We prefer `LiteAnalyzer` from `birdnetlib`. If unavailable, we fall back to `Analyzer`.


In [3]:
# Teacher: full BirdNET (desktop) — reliable on Windows
from birdnetlib.analyzer import Analyzer
from birdnetlib import Recording

analyzer = Analyzer()
TEACHER_KIND = "Analyzer (full BirdNET)"
print("Teacher ready:", TEACHER_KIND)


c:\Users\shado\Year3Projects\FYP\.venv\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model


c:\Users\shado\Year3Projects\FYP\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Meta model loaded.
Teacher ready: Analyzer (full BirdNET)


In [4]:
from birdnetlib import Recording

## 3) Utilities


In [5]:
import math
import numpy as np
import librosa
import soundfile as sf
import tempfile
import random

def rms_dbfs(x: np.ndarray) -> float:
    rms = float(np.sqrt(np.mean(np.square(x)) + EPS))
    return float(20.0 * math.log10(rms + EPS))

def window_starts(region_len_s: float) -> list[float]:
    starts = []
    if INCLUDE_ZERO_WINDOW:
        starts.append(0.0)
    s = SKIP_FIRST_S
    while s + CLIP_LEN_S <= region_len_s + 1e-9:
        starts.append(float(s))
        s += STRIDE_S
    return sorted(set(starts))

def load_audio_segment(path: Path, sr: int, offset_s: float, duration_s: float) -> np.ndarray:
    y, _ = librosa.load(str(path), sr=sr, mono=True, offset=float(offset_s), duration=float(duration_s))
    return y

def choose_best_region(path: Path, total_len_s: float) -> tuple[float, np.ndarray]:
    """
    If audio <= CAP_RECORDING_S: use full audio
    Else: scan start offsets every CAP_STEP_S and pick the CAP_RECORDING_S region with highest RMS.
    """
    if total_len_s <= CAP_RECORDING_S + 1e-9:
        y = load_audio_segment(path, SR_MODEL, 0.0, total_len_s)
        return 0.0, y

    best_start, best_score, best_y = 0.0, -1e9, None
    max_start = max(0.0, total_len_s - CAP_RECORDING_S)

    for s in np.arange(0.0, max_start + 1e-9, CAP_STEP_S):
        y = load_audio_segment(path, SR_MODEL, float(s), CAP_RECORDING_S)
        score = rms_dbfs(y)
        if score > best_score:
            best_start, best_score, best_y = float(s), score, y

    assert best_y is not None
    return best_start, best_y

import os
import tempfile
import soundfile as sf

def teacher_analyze_window(y_16k: np.ndarray, target_sci_name: str) -> dict:
    if analyzer is None:
        raise RuntimeError("Teacher analyzer not initialized.")

    y_48k = librosa.resample(y_16k, orig_sr=SR_MODEL, target_sr=SR_TEACHER)

    fd, tmp_path = tempfile.mkstemp(suffix=".wav")  # <-- Windows-safe
    os.close(fd)  # important: release the handle

    try:
        sf.write(tmp_path, y_48k, SR_TEACHER, subtype="PCM_16")

        rec = Recording(analyzer, tmp_path, min_conf=BIRD_CONF_THR)
        rec.analyze()
        dets = rec.detections or []

    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass

    # --- rest unchanged ---
    top = None
    max_conf = 0.0
    for d in dets:
        c = float(d.get("confidence", 0.0))
        if c > max_conf:
            max_conf, top = c, d

    top_sci = (top.get("scientific_name") if top else "")
    top_common = (top.get("common_name") if top else "")
    top_conf = float(top.get("confidence", 0.0)) if top else 0.0

    target_norm = target_sci_name.strip().lower()
    is_target = bool(top_sci) and (top_sci.strip().lower() == target_norm) and (top_conf >= SPECIES_CONF_THR)

    if len(dets) == 0:
        decision = "non_bird"
    elif is_target:
        decision = "species"
    else:
        decision = "drop"

    return {
        "detections": dets,
        "top_sci": top_sci,
        "top_common": top_common,
        "top_conf": top_conf,
        "max_conf": float(max_conf),
        "decision": decision,
    }


## 4) Process one species


In [6]:
import pandas as pd
from tqdm.auto import tqdm


c:\Users\shado\Year3Projects\FYP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import re

def save_clip(y_16k: np.ndarray, out_dir: Path, xc_id: str, start_s: float, end_s: float) -> str:
    start_ms = int(round(start_s * 1000))
    end_ms = int(round(end_s * 1000))
    fname = f"XC{xc_id}__s{start_ms}__e{end_ms}.wav"
    out_path = out_dir / fname
    out_dir.mkdir(parents=True, exist_ok=True)
    sf.write(str(out_path), y_16k, SR_MODEL, subtype="PCM_16")
    return str(out_path)


def process_species(species_label: str, max_recordings: int | None = None) -> pd.DataFrame:
    in_csv = MANIFESTS_DIR / f"{species_label}_downloaded.csv"
    if not in_csv.exists():
        raise FileNotFoundError(f"Missing: {in_csv}")

    df = pd.read_csv(in_csv)
    if max_recordings is not None:
        df = df.head(max_recordings).copy()

    out_rows = []

    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"{species_label}"):
        xc_id = str(r["xc_id"])
        sci_name = str(r["sci_name"])
        src_path = resolve_source_path(str(r["local_path"]))

        if not src_path.exists():
            out_rows.append({"species_label": species_label, "xc_id": xc_id, "error": "missing_source"})
            continue

        # Duration
        try:
            total_len_s = float(librosa.get_duration(path=str(src_path)))
        except Exception:
            y_tmp, _ = librosa.load(str(src_path), sr=SR_MODEL, mono=True)
            total_len_s = float(len(y_tmp) / SR_MODEL)

        # Choose best region (cap long recordings)
        region_start_s, y_region = choose_best_region(src_path, total_len_s)
        region_len_s = float(len(y_region) / SR_MODEL)

        starts = window_starts(region_len_s)
        if not starts:
            out_rows.append({"species_label": species_label, "xc_id": xc_id, "error": "no_windows"})
            continue

        # Candidate windows
        candidates = []
        for s in starts:
            s_i = int(round(s * SR_MODEL))
            e_i = s_i + int(round(CLIP_LEN_S * SR_MODEL))
            if e_i > len(y_region):
                continue
            w = y_region[s_i:e_i]
            candidates.append({
                "start_s": float(region_start_s + s),
                "end_s": float(region_start_s + s + CLIP_LEN_S),
                "rms_db": rms_dbfs(w),
                "wave_16k": w,
            })

        if not candidates:
            out_rows.append({"species_label": species_label, "xc_id": xc_id, "error": "no_candidates"})
            continue

        # RMS gate (filter only)
        rms_vals = np.array([c["rms_db"] for c in candidates], dtype=float)
        thr = max(RMS_ABS_MIN_DB, float(np.percentile(rms_vals, RMS_KEEP_PERCENTILE)))
        gated = [c for c in candidates if c["rms_db"] >= thr]
        if not gated:
            gated = [candidates[int(np.argmax(rms_vals))]]

        # Teacher classify
        for c in gated:
            t = teacher_analyze_window(c["wave_16k"], sci_name)
            c.update({
                "teacher_decision": t["decision"],
                "teacher_top_sci": t["top_sci"],
                "teacher_top_common": t["top_common"],
                "teacher_top_conf": t["top_conf"],
                "teacher_max_conf": t["max_conf"],
            })

        species_pos = [c for c in gated if c["teacher_decision"] == "species"]
        nonbird = [c for c in gated if c["teacher_decision"] == "non_bird"]

        # Seeded random selection for positives
        rng = random.Random(SEED + int(xc_id))
        rng.shuffle(species_pos)
        sel_species = species_pos[:MAX_SPECIES_CLIPS_PER_REC]

        # Non-bird: select lowest teacher_max_conf (most confidently non-bird)
        nonbird_sorted = sorted(nonbird, key=lambda x: x["teacher_max_conf"])
        sel_nonbird = nonbird_sorted[:NONBIRD_CLIPS_PER_REC]

        out_species_dir = OUT_SPECIES_DIR / species_label
        out_nonbird_dir = OUT_NONBIRD_DIR / species_label

        selected_set = set((c["start_s"], c["end_s"]) for c in (sel_species + sel_nonbird))

        for c in gated:
            key = (c["start_s"], c["end_s"])
            selected = int(key in selected_set)
            clip_path = ""
            selected_reason = ""

            if selected:
                if c in sel_species:
                    clip_path = save_clip(c["wave_16k"], out_species_dir, xc_id, c["start_s"], c["end_s"])
                    selected_reason = "rand_species"
                    final_class = "species"
                else:
                    clip_path = save_clip(c["wave_16k"], out_nonbird_dir, xc_id, c["start_s"], c["end_s"])
                    selected_reason = "nonbird_lowconf"
                    final_class = "non_bird"
            else:
                final_class = c["teacher_decision"]

            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(src_path),
                "start_s": c["start_s"],
                "end_s": c["end_s"],
                "rms_db": c["rms_db"],
                "rms_gate_thr_db": thr,
                "teacher_decision": c["teacher_decision"],
                "teacher_top_sci": c.get("teacher_top_sci", ""),
                "teacher_top_common": c.get("teacher_top_common", ""),
                "teacher_top_conf": c.get("teacher_top_conf", 0.0),
                "teacher_max_conf": c.get("teacher_max_conf", 0.0),
                "final_class": final_class,
                "selected": selected,
                "selected_reason": selected_reason,
                "clip_path": clip_path,
                "error": "",
            })

    out_df = pd.DataFrame(out_rows)
    out_csv = OUT_CLIP_MANIFESTS_DIR / f"{species_label}_clips.csv"
    out_df.to_csv(out_csv, index=False)
    print("Wrote:", out_csv)
    return out_df




## 5) Batch over all downloaded species


In [ ]:
def list_downloaded_species() -> list[str]:
    files = sorted(MANIFESTS_DIR.glob("*_downloaded.csv"))
    species = []
    for f in files:
        m = re.match(r"(.+)_downloaded\.csv$", f.name)
        if m:
            species.append(m.group(1))
    return species

def process_all_species(max_species: int | None = None, max_recordings_per_species: int | None = None) -> pd.DataFrame:
    species_list = list_downloaded_species()
    if max_species is not None:
        species_list = species_list[:max_species]

    summaries = []
    for sp in species_list:
        df_sp = process_species(sp, max_recordings=max_recordings_per_species)
        sel = df_sp[df_sp["selected"] == 1]
        summaries.append({
            "species_label": sp,
            "selected_total": int(len(sel)),
            "selected_species": int((sel["final_class"] == "species").sum()),
            "selected_nonbird": int((sel["final_class"] == "non_bird").sum()),
            "unique_recordings": int(df_sp["xc_id"].nunique()),
        })

    sum_df = pd.DataFrame(summaries).sort_values("species_label")
    sum_csv = OUT_CLIP_MANIFESTS_DIR / "clips_v1_summary.csv"
    sum_df.to_csv(sum_csv, index=False)
    print("Wrote:", sum_csv)
    return sum_df

# Start small:
summary = process_all_species(max_species=10, max_recordings_per_species=None)
# summary


accipiter_nisus:   0%|          | 0/5 [00:00<?, ?it/s]

accipiter_nisus:  20%|██        | 1/5 [00:08<00:34,  8.71s/it]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpn2100dtj.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpke9tezr3.wav


accipiter_nisus:  40%|████      | 2/5 [00:08<00:10,  3.65s/it]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpnje65w70.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmp3jeti2hx.wav


...

read_audio_data: complete, read  1 chunks.
analyze_recording tmptht5zemc.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpwy9j4o4t.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmp1en47_wc.wav


cinclus_cinclus:  80%|████████  | 4/5 [00:01<00:00,  2.46it/s]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmp7dskybn4.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmphzn72djj.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpjdilkjsl.wav


cinclus_cinclus: 100%|██████████| 5/5 [00:01<00:00,  2.60it/s]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpp2a0nsxl.wav
Wrote: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\manifests\clips_v1\cinclus_cinclus_clips.csv
Wrote: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\manifests\clips_v1\clips_v1_summary.csv


## 6) Notes / next steps

- **Train/val/test split:** split by `xc_id` (recording), not by clip file.
- **Tune thresholds:** `bird_conf_thr`, `species_conf_thr`, and RMS gate settings.
- **Spectrogram notebook** should read from:
  - `bird_data/clips_v1/species/<species>/...` and `bird_data/clips_v1/non_bird/<species>/...`


In [10]:
process_species("corvus_corax", max_recordings=None)

corvus_corax:   0%|          | 0/250 [00:00<?, ?it/s]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmp5oc3yb0h.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpjj9pp2zu.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpft2dn364.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpfbjwmj63.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmp871jm6gg.wav


corvus_corax:   0%|          | 1/250 [00:01<04:28,  1.08s/it]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpgt98u8mr.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpzjpz4mu7.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpcrw_n9ok.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpy7dsxf0j.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpk2bym38w.wav


corvus_corax:   1%|          | 2/250 [00:01<02:39,  1.55it/s]

...

corvus_corax: 100%|█████████▉| 249/250 [01:47<00:00,  3.58it/s]

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmprdbq61er.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording tmpag717_zz.wav


corvus_corax: 100%|██████████| 250/250 [01:47<00:00,  2.33it/s]

Wrote: C:\Users\shado\Year3Projects\FYP\src\dataset\bird_data\manifests\clips_v1\corvus_corax_clips.csv


,species_label,xc_id,sci_name,source_path,start_s,end_s,rms_db,rms_gate_thr_db,teacher_decision,teacher_top_sci,teacher_top_common,teacher_top_conf,teacher_max_conf,final_class,selected,selected_reason,clip_path,error
0,corvus_corax,864746,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,0.0,3.0,-17.825348,-19.236449,non_bird,,,0.000000,0.000000,non_bird,1,nonbird_lowconf,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,
1,corvus_corax,864746,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,1.0,4.0,-16.799145,-19.236449,non_bird,,,0.000000,0.000000,non_bird,1,nonbird_lowconf,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,
2,corvus_corax,864746,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,4.0,7.0,-17.759498,-19.236449,species,Corvus corax,Common Raven,0.647615,0.647615,species,1,rand_species,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,
3,corvus_corax,864746,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,13.0,16.0,-16.585269,-19.236449,non_bird,,,0.000000,0.000000,non_bird,0,,,
4,corvus_corax,864746,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,16.0,19.0,-16.328326,-19.236449,non_bird,,,0.000000,0.000000,non_bird,0,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,corvus_corax,675087,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,22.0,25.0,-38.006004,-40.000000,species,Corvus corax,Common Raven,0.446526,0.446526,species,1,rand_species,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,
944,corvus_corax,958991,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,7.0,10.0,-43.645194,-40.000000,species,Corvus corax,Common Raven,0.386937,0.386937,species,1,rand_species,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,
945,corvus_corax,833106,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,1.0,4.0,-32.186029,-32.238574,species,Corvus corax,Common Raven,0.881560,0.881560,species,1,rand_species,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,
946,corvus_corax,833106,Corvus corax,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,4.0,7.0,-31.455312,-32.238574,non_bird,,,0.000000,0.000000,non_bird,1,nonbird_lowconf,C:\Users\shado\Year3Projects\FYP\src\dataset\b...,


In [11]:
df = pd.read_csv(OUT_CLIP_MANIFESTS_DIR / "corvus_corax_clips.csv")
print("rows:", len(df))
print("unique recordings:", df["xc_id"].nunique())
print(df["teacher_decision"].value_counts())
print("avg windows per recording:", len(df) / df["xc_id"].nunique())


rows: 948
unique recordings: 250
teacher_decision
species     683
drop        137
non_bird    128
Name: count, dtype: int64
avg windows per recording: 3.792
